In [1]:
%pip install -U imbalanced-learn
%load_ext autoreload
%autoreload 2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from imblearn.over_sampling import SMOTENC, SMOTEN
from imblearn.combine import SMOTETomek
from sklearn.tree import DecisionTreeClassifier
from sklearn.compose import ColumnTransformer

from sklearn.metrics import fbeta_score,make_scorer
from custom_func import feature_engineering_light, encode_categorical_features,find_best_threshold_f2
from sklearn.model_selection import GridSearchCV, train_test_split,RandomizedSearchCV
from sklearn.metrics import roc_curve, auc, f1_score, confusion_matrix, roc_auc_score, r2_score, mean_squared_error,precision_recall_curve
from sklearn.metrics import classification_report, precision_score, recall_score
from sklearn.multiclass import OneVsRestClassifier, OneVsOneClassifier


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [ ]:
df = pd.read_csv('../data/raw/bank-additional-full.csv', sep=';')
df_train,df_test = train_test_split(df, test_size=0.2, random_state=42, stratify=df['y'])
df_train,df_val = train_test_split(df_train, test_size=0.25, random_state=42, stratify=df_train['y'])

In [ ]:
df_train = feature_engineering_light(df_train)
df_val = feature_engineering_light(df_val)
df_test = feature_engineering_light(df_test)

df_train['y'] = df_train['y'].map({'yes': 1, 'no': 0}).astype(int)
df_val['y'] = df_val['y'].map({'yes': 1, 'no': 0}).astype(int)
df_test['y'] = df_test['y'].map({'yes': 1, 'no': 0}).astype(int)

categorical_cols = df_train.select_dtypes(exclude=np.number).columns.tolist()
df_train,df_val,df_test,encoded_cols = encode_categorical_features(df_train, df_val, df_test, categorical_cols)

df_train.head()

Drop: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'day_of_week', 'poutcome', 'season', 'age_group']


,duration,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y,job_admin.,job_blue-collar,job_entrepreneur,job_housemaid,job_management,job_retired,job_self-employed,job_services,job_student,job_technician,job_unemployed,job_unknown,marital_divorced,marital_married,marital_single,marital_unknown,education_basic.4y,education_basic.6y,education_basic.9y,education_high.school,education_professional.course,education_university.degree,education_unknown,default_no,default_unknown,default_yes,housing_no,housing_unknown,housing_yes,loan_no,loan_unknown,loan_yes,contact_cellular,contact_telephone,day_of_week_fri,day_of_week_mon,day_of_week_thu,day_of_week_tue,day_of_week_wed,poutcome_failure,poutcome_nonexistent,poutcome_success,season_autumn,season_spring,season_summer,season_winter,age_group_Adult,age_group_Child,age_group_Middle Age,age_group_Senior,age_group_Young Adult
7302,101,23,999,0,1.10,93.99,-36.40,4.86,5191.00,0,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00
6693,217,2,999,0,1.10,93.99,-36.40,4.86,5191.00,0,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00
40376,153,1,0,5,-1.70,94.03,-38.30,0.90,4991.60,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,1.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,1.00
24387,69,1,999,0,-0.10,93.20,-42.00,4.19,5195.80,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00
33177,631,1,11,1,-1.80,92.89,-46.20,1.29,5099.10,0,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,1.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,1.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00


In [4]:

X_train = df_train.drop(columns=['y'])
X_val = df_val.drop(columns=['y'])
X_test = df_test.drop(columns=['y'])


y_train = df_train['y']
y_val = df_val['y']
y_test = df_test['y']


In [5]:
'''
 param_grid = {
    'max_depth': [3, 5, 7, 10, None],
    'min_samples_leaf': [10, 20, 50, 100],
    'min_samples_split': [10, 20, 50],
    'class_weight': [None, 'balanced']
}

f2_scorer = make_scorer(fbeta_score, beta=2)

grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid,
    scoring=f2_scorer,
    cv=5,
    n_jobs=-1
)

grid.fit(X_train, y_train)

print(grid.best_params_)
print(grid.best_score_)
'''

"\n param_grid = {\n    'max_depth': [3, 5, 7, 10, None],\n    'min_samples_leaf': [10, 20, 50, 100],\n    'min_samples_split': [10, 20, 50],\n    'class_weight': [None, 'balanced']\n}\n\nf2_scorer = make_scorer(fbeta_score, beta=2)\n\ngrid = GridSearchCV(\n    DecisionTreeClassifier(random_state=42),\n    param_grid,\n    scoring=f2_scorer,\n    cv=5,\n    n_jobs=-1\n)\n\ngrid.fit(X_train, y_train)\n\nprint(grid.best_params_)\nprint(grid.best_score_)\n"

In [6]:
model = DecisionTreeClassifier(random_state=42,max_depth=7, min_samples_leaf=7, min_samples_split=10,class_weight='balanced')
model.fit(X_train, y_train)

print("Depth of the tree:", model.get_depth())

Depth of the tree: 7


In [7]:
prob_val = model.predict_proba(X_val)[:,1]
best_t, best_f2, best_recall, best_precision = find_best_threshold_f2(y_val, prob_val, beta=2)
print("Best threshold:", best_t)
print("F2-score at this threshold:", best_f2)
print("Recall at this threshold:", best_recall)
print("Precision at this threshold:", best_precision)

Best threshold: 0.6714285714285715
F2-score at this threshold: 0.7468841642228738
Recall at this threshold: 0.8782327586206896
Precision at this threshold: 0.4673165137614679


In [8]:
# ===== TRAIN =====
prob_train = model.predict_proba(X_train)[:,1]
pred_train_thresh = (prob_train >= best_t).astype(int)  # використання твого threshold

precision_train = precision_score(y_train, pred_train_thresh)
recall_train = recall_score(y_train, pred_train_thresh)
f2_train = fbeta_score(y_train, pred_train_thresh, beta=2)

# ===== VAL =====
prob_val = model.predict_proba(X_val)[:,1]
pred_val_thresh = (prob_val >= best_t).astype(int)

precision_val = precision_score(y_val, pred_val_thresh)
recall_val = recall_score(y_val, pred_val_thresh)
f2_val = fbeta_score(y_val, pred_val_thresh, beta=2)


results = {
    'F2_train_val': f"Train:{round(f2_train,2)} Val:{round(f2_val,2)}",
    'Precision': f"Train:{round(precision_train,2)} Val:{round(precision_val,2)}",
    'Recall': f"Train:{round(recall_train,2)} Val:{round(recall_val,2)}",
    
    'Diff_F2': round(f2_train - f2_val, 2),
    'Diff_Precision': round(precision_train - precision_val, 2),
    'Diff_Recall': round(recall_train - recall_val, 2),
}

results

{'F2_train_val': 'Train:0.76 Val:0.75',
 'Precision': 'Train:0.48 Val:0.47',
 'Recall': 'Train:0.89 Val:0.88',
 'Diff_F2': 0.01,
 'Diff_Precision': 0.01,
 'Diff_Recall': 0.01}

In [9]:

# ===== TEST =====
prob_test = model.predict_proba(X_test)[:,1]
pred_test_thresh = (prob_test >= best_t).astype(int)

precision_test = precision_score(y_test, pred_test_thresh)
recall_test = recall_score(y_test, pred_test_thresh)
f2_test = fbeta_score(y_test, pred_test_thresh, beta=2)


results = {
    'F2_test': f"TEST:{round(f2_test,2)}",
    'Precision': f"TEST:{round(precision_test,2)}",
    'Recall': f"TEST:{round(recall_test,2)}",
    
}

results

{'F2_test': 'TEST:0.74', 'Precision': 'TEST:0.47', 'Recall': 'TEST:0.87'}